# 6주차 — **머신러닝**(ML) **워크플로우**(Workflow): **전처리**(Preprocessing)와 **교차 검증**(Cross-Validation)

---

## 학습 목표

1. **머신러닝**(Machine Learning)의 전체 **파이프라인**(Pipeline)을 이해한다.
2. **전처리**(Preprocessing) 기법 — **스케일링**(Scaling), **인코딩**(Encoding), **결측값 처리**(Imputation) — 을 익힌다.
3. **교차 검증**(Cross-Validation)의 원리와 필요성을 이해하고 구현한다.
4. **심부전증**(Heart Failure) 예측 데이터를 활용하여 실전 ML **워크플로우**를 완성한다.

---

## 활용 데이터

**심부전증 예측 데이터**(Heart Failure Clinical Records Dataset)  
→ 출처: [Kaggle](https://www.kaggle.com/andrewmvd/heart-failure-clinical-data)

| 컬럼(Column) | 설명(Description) | 타입(Type) |
|---|---|---|
| `age` | 환자의 나이 | 수치형 |
| `anaemia` | 빈혈증 여부 (0: 정상, 1: 빈혈) | 범주형 |
| `creatinine_phosphokinase` | 크레아틴키나제 검사 결과 | 수치형 |
| `diabetes` | 당뇨병 여부 (0: 정상, 1: 당뇨) | 범주형 |
| `ejection_fraction` | 박출계수 (%) | 수치형 |
| `high_blood_pressure` | 고혈압 여부 (0: 정상, 1: 고혈압) | 범주형 |
| `platelets` | 혈소판 수 (kiloplatelets/mL) | 수치형 |
| `serum_creatinine` | 혈중 크레아틴 레벨 (mg/dL) | 수치형 |
| `serum_sodium` | 혈중 나트륨 레벨 (mEq/L) | 수치형 |
| `sex` | 성별 (0: 여성, 1: 남성) | 범주형 |
| `smoking` | 흡연 여부 (0: 비흡연, 1: 흡연) | 범주형 |
| `time` | 관찰 기간 (일) | 수치형 |
| `DEATH_EVENT` | 사망 여부 (0: 생존, 1: 사망) → **타겟 변수** | 범주형 |

---

# PART 1: **머신러닝**(Machine Learning) 개요

---

## 1.1 **머신러닝**(Machine Learning)이란?

**머신러닝**은 명시적으로 프로그래밍하지 않아도, 데이터로부터 패턴을 학습하여 예측이나 의사결정을 수행하는 알고리즘이다.

전통적인 프로그래밍과 머신러닝의 차이를 비교하면 다음과 같다:

```
전통적 프로그래밍(Traditional Programming)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [데이터] + [규칙(사람이 작성)] ──→ [결과]

머신러닝(Machine Learning)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [데이터] + [결과(정답)] ──→ [규칙(기계가 학습)]
```

예를 들어, 심부전증 환자의 사망 여부를 예측한다고 하자. 전통적 방법에서는 의사가 "나이 > 70이고 박출계수 < 30이면 위험"이라는 규칙을 직접 만들어야 한다. 머신러닝에서는 과거 환자 데이터(나이, 검사 결과, 사망 여부)를 주면 기계가 스스로 어떤 조합이 위험한지 학습한다.

## 1.2 **머신러닝**의 세 가지 유형

```
┌─────────────────────────────────────────────────────────────────┐
│                     머신러닝(Machine Learning)                    │
├─────────────────┬──────────────────┬────────────────────────────┤
│  지도학습         │  비지도학습        │  강화학습                    │
│  (Supervised)    │  (Unsupervised)   │  (Reinforcement)           │
├─────────────────┼──────────────────┼────────────────────────────┤
│  정답(Label)이    │  정답 없이         │  보상(Reward) 신호를 통해   │
│  주어진 데이터로  │  데이터의 구조를   │  최적의 행동 전략을         │
│  학습한다         │  발견한다          │  학습한다                   │
├─────────────────┼──────────────────┼────────────────────────────┤
│  → 분류, 회귀     │  → 군집화, 차원축소 │  → 게임 AI, 로봇 제어      │
│  예) 스팸 분류    │  예) 고객 세분화    │  예) AlphaGo               │
│  예) 집값 예측    │  예) 이상 탐지      │  예) 자율주행               │
└─────────────────┴──────────────────┴────────────────────────────┘
```

이번 6주차에서는 **지도학습**(Supervised Learning) 중 **분류**(Classification)를 다룬다. 심부전증 데이터에서 `DEATH_EVENT`(사망 여부)라는 정답이 있으므로, 이는 전형적인 지도학습 분류 문제이다.

## 1.3 **ML 워크플로우**(Workflow): 전체 파이프라인

머신러닝 프로젝트는 다음 6단계를 순차적으로 수행한다:

```
[1단계]        [2단계]        [3단계]         [4단계]        [5단계]        [6단계]
문제 정의  ──→  데이터 수집  ──→  데이터 전처리  ──→  모델 학습  ──→  모델 평가  ──→  배포
(Problem)      (Collect)      (Preprocess)     (Train)       (Evaluate)     (Deploy)
   │              │               │                │              │              │
   │              │               │                │              │              │
   ▼              ▼               ▼                ▼              ▼              ▼
 "심부전증     "Kaggle에서      "결측값 처리,     "Logistic     "정확도,       "병원 시스템에
  사망 예측"    CSV 다운로드"    스케일링,        Regression     F1 Score,      통합"
                                인코딩"          학습"          ROC-AUC"
```

각 단계를 요리에 비유하면 이렇다:
- 1단계(문제 정의): "어떤 요리를 만들까?" 결정
- 2단계(데이터 수집): 재료 장보기
- **3단계(전처리): 재료 손질 — 씻고, 자르고, 계량하기** ← 오늘의 핵심
- 4단계(모델 학습): 요리하기
- **5단계(모델 평가): 맛보기** ← 오늘의 핵심
- 6단계(배포): 손님에게 서빙

실무에서 데이터 과학자는 전체 시간의 **60~80%를 3단계(전처리)**에 쓴다. 재료 손질이 허술하면 아무리 좋은 레시피(알고리즘)를 써도 맛있는 요리가 나올 수 없는 것과 같다.

---

## 1.4 **전처리**(Preprocessing): 데이터를 모델이 이해할 수 있는 형태로 변환하기

전처리의 핵심 작업 3가지를 정리하면 다음과 같다:

### 1.4.1 **결측값 처리**(Imputation)

결측값(Missing Value, `NaN`)은 데이터가 비어 있는 것이다. 대부분의 ML 알고리즘은 결측값을 처리하지 못하므로, 학습 전에 반드시 처리해야 한다.

```
처리 방법                    설명                              사용 시점
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
제거(Drop)                  결측값이 있는 행 또는 열 삭제       결측 비율 < 5%일 때
평균/중앙값 대체(Mean/Med)   수치형 변수의 결측을 평균으로      이상치가 적을 때 평균,
                                                              이상치가 많으면 중앙값
최빈값 대체(Mode)            범주형 변수의 결측을 최빈값으로    범주형 변수일 때
```

### 1.4.2 **스케일링**(Scaling)

서로 다른 단위와 범위를 가진 특성(Feature)들을 동일한 척도로 맞추는 작업이다.

```
예시: 심부전증 데이터의 원본 범위
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  age:                 40 ~ 95           (범위: 55)
  creatinine_phospho:  23 ~ 7,861       (범위: 7,838)
  platelets:           25,100 ~ 850,000 (범위: 824,900)
  serum_sodium:        113 ~ 148        (범위: 35)

→ 범위가 크게 다르면, 큰 값을 가진 특성이 모델에 과도한 영향을 미친다!
→ 스케일링으로 모든 특성을 비슷한 범위로 맞춘다.
```

주요 스케일링 방법 2가지:

```
StandardScaler (표준화)                    MinMaxScaler (정규화)
━━━━━━━━━━━━━━━━━━━━━━━━━━              ━━━━━━━━━━━━━━━━━━━━━━━
z = (x - 평균) / 표준편차                  x_scaled = (x - 최솟값) / (최댓값 - 최솟값)

→ 평균 = 0, 표준편차 = 1                   → 0 ~ 1 범위로 변환
→ 이상치에 상대적으로 강건                  → 이상치에 민감
→ 대부분의 ML 알고리즘에 권장               → 신경망(Neural Network)에서 자주 사용
```

### 1.4.3 **인코딩**(Encoding)

머신러닝 알고리즘은 숫자만 처리할 수 있다. 따라서 범주형(Categorical) 데이터를 숫자로 변환해야 한다.

```
LabelEncoder (레이블 인코딩)              OneHotEncoder (원-핫 인코딩)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━           ━━━━━━━━━━━━━━━━━━━━━━━━━━━
범주 → 정수                               범주 → 이진 벡터

예) 혈액형:                               예) 혈액형:
  A형 → 0                                  A형 → [1, 0, 0, 0]
  B형 → 1                                  B형 → [0, 1, 0, 0]
  O형 → 2                                  O형 → [0, 0, 1, 0]
  AB형 → 3                                 AB형 → [0, 0, 0, 1]

→ 순서가 있는 범주에 적합                  → 순서가 없는 범주에 적합
  (예: 학점 A > B > C)                       (예: 혈액형, 색상)
→ 주의: 숫자 크기에 의미가               → 특성 수가 크게 늘어날 수 있다
  부여될 수 있다 (A < AB?)
```

심부전증 데이터에서는 `anaemia`, `diabetes`, `sex` 등이 이미 0/1로 인코딩되어 있으므로 추가 인코딩이 필요 없다. 그러나 일반적인 데이터셋에서는 반드시 확인해야 한다.

---

## 1.5 **교차 검증**(Cross-Validation): 모델을 공정하게 평가하는 방법

### 왜 교차 검증이 필요한가?

`train_test_split`으로 한 번만 나누면, 어떤 데이터가 학습에 들어가고 어떤 데이터가 테스트에 들어가느냐에 따라 결과가 크게 달라질 수 있다. 이를 **운에 의한 평가**라고 한다.

```
train_test_split 한 번만 사용했을 때의 문제:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  1회차: 정확도 = 0.85  ← "꽤 좋네!"
  2회차: 정확도 = 0.72  ← "어? 왜 다르지?"     ← random_state가 다르면
  3회차: 정확도 = 0.91  ← "이게 진짜인가?"         결과가 매번 달라진다!

→ 어느 것이 이 모델의 "진짜 성능"인가?
```

### **K-Fold 교차 검증**(K-Fold Cross-Validation)

데이터를 K개의 동일한 크기 그룹(Fold)으로 나누고, 각 그룹이 한 번씩 테스트 세트가 되도록 K번 반복 학습-평가한다.

```
5-Fold 교차 검증 예시 (K=5):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Fold 1: [테스트] [학 습] [학 습] [학 습] [학 습]  → 정확도 = 0.82
 Fold 2: [학 습] [테스트] [학 습] [학 습] [학 습]  → 정확도 = 0.87
 Fold 3: [학 습] [학 습] [테스트] [학 습] [학 습]  → 정확도 = 0.84
 Fold 4: [학 습] [학 습] [학 습] [테스트] [학 습]  → 정확도 = 0.79
 Fold 5: [학 습] [학 습] [학 습] [학 습] [테스트]  → 정확도 = 0.86

 최종 성능 = 평균(0.82, 0.87, 0.84, 0.79, 0.86) = 0.836 ± 0.028
                                                     ↑           ↑
                                                   평균 성능    안정성(분산)
```

### **Stratified K-Fold**(계층적 K-폴드)

일반 K-Fold와 동일하나, 각 Fold에서 **타겟 변수의 비율이 원본과 동일하게 유지**된다. 심부전증 데이터처럼 사망 비율(약 32%)이 불균형한 경우, Stratified K-Fold를 사용해야 한다.

```
원본 데이터: 사망 32% / 생존 68%

일반 K-Fold (비율 보장 안 됨)        Stratified K-Fold (비율 유지)
━━━━━━━━━━━━━━━━━━━━━━━━━━         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Fold 1: 사망 45% / 생존 55%  ❌     Fold 1: 사망 32% / 생존 68%  ✅
 Fold 2: 사망 20% / 생존 80%  ❌     Fold 2: 사망 32% / 생존 68%  ✅
 Fold 3: 사망 31% / 생존 69%  ✅     Fold 3: 사망 32% / 생존 68%  ✅
```

---

# PART 2: **가이드 실습**(Guided Practice) — Scikit-learn 핵심 API

---

## 2.1 Scikit-learn의 핵심 패턴: `fit()` → `transform()` → `predict()`

Scikit-learn의 모든 도구는 동일한 패턴을 따른다:

```
┌──────────────────────────────────────────────────────────────┐
│  전처리 도구 (Scaler, Encoder)                                │
│  ─────────────────────────────                                │
│  1. fit(data)       → 데이터에서 통계량(평균, 표준편차 등) 학습 │
│  2. transform(data) → 학습한 통계량으로 데이터 변환             │
│  3. fit_transform() → 1 + 2를 한 번에 수행                    │
│                                                               │
│  모델 도구 (Classifier, Regressor)                             │
│  ─────────────────────────────────                             │
│  1. fit(X, y)       → 학습 데이터로 모델 훈련                  │
│  2. predict(X)      → 새로운 데이터에 대해 예측                 │
│  3. score(X, y)     → 예측 성능(정확도 등) 계산                 │
└──────────────────────────────────────────────────────────────┘
```

## 2.2 실습: `StandardScaler` 사용법

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# 예제 데이터: 환자 3명의 나이와 혈소판 수
data = np.array([
    [40, 250000],   # 환자 A
    [65, 130000],   # 환자 B
    [80, 380000]    # 환자 C
])

print("=== 원본 데이터 ===")
print(f"  나이 범위:   {data[:, 0].min()} ~ {data[:, 0].max()}")
print(f"  혈소판 범위: {data[:, 1].min():,.0f} ~ {data[:, 1].max():,.0f}")
print()

# StandardScaler 적용
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

print("=== 스케일링 후 데이터 ===")
print(f"  나이 범위:   {data_scaled[:, 0].min():.2f} ~ {data_scaled[:, 0].max():.2f}")
print(f"  혈소판 범위: {data_scaled[:, 1].min():.2f} ~ {data_scaled[:, 1].max():.2f}")
print()
print(f"  나이 평균: {data_scaled[:, 0].mean():.6f}  (0에 가까워야 한다)")
print(f"  나이 표준편차: {data_scaled[:, 0].std():.6f}  (1에 가까워야 한다)")

## 2.3 실습: `train_test_split` 사용법

In [ ]:
from sklearn.model_selection import train_test_split

# 간단한 예제 데이터
X = np.array([[1], [2], [3], [4], [5], [6], [7], [8], [9], [10]])
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

# 학습/테스트 분리 (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== 데이터 분리 결과 ===")
print(f"  전체 데이터: {len(X)}개")
print(f"  학습 데이터: {len(X_train)}개 ({len(X_train)/len(X)*100:.0f}%)")
print(f"  테스트 데이터: {len(X_test)}개 ({len(X_test)/len(X)*100:.0f}%)")
print()
print(f"  학습 세트 타겟 비율: 0={sum(y_train==0)}, 1={sum(y_train==1)}")
print(f"  테스트 세트 타겟 비율: 0={sum(y_test==0)}, 1={sum(y_test==1)}")
print()
print("→ stratify=y 덕분에 학습/테스트 모두 0과 1의 비율이 유지된다.")

## 2.4 실습: `cross_val_score` 사용법

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

# 간단한 예제로 교차 검증 이해
from sklearn.datasets import make_classification

X_demo, y_demo = make_classification(
    n_samples=200, n_features=5, random_state=42
)

model = LogisticRegression(max_iter=1000, random_state=42)

# 5-Fold 교차 검증
scores = cross_val_score(model, X_demo, y_demo, cv=5, scoring='accuracy')

print("=== 5-Fold 교차 검증 결과 ===")
for i, score in enumerate(scores, 1):
    print(f"  Fold {i}: 정확도 = {score:.4f}")
print(f"\n  평균 정확도: {scores.mean():.4f}")
print(f"  표준편차:    {scores.std():.4f}")
print(f"\n  → 최종 보고: {scores.mean():.4f} ± {scores.std():.4f}")

---

# PART 3: **프로젝트 실습** — "데이터 분석으로 **심부전증**을 예방할 수 있을까?"

---

## Step 1. **데이터셋 준비**(Dataset Preparation)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Colab용)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

### 문제 1. 데이터 불러오기 및 기본 구조 파악

심부전증 데이터셋(`heart_failure_clinical_records_dataset.csv`)을 불러와서 기본 구조를 파악하라.

In [ ]:
# 데이터 불러오기 (Kaggle 또는 직접 업로드)
# df = pd.read_csv('heart_failure_clinical_records_dataset.csv')

# Kaggle API 사용 시:
# !pip install kaggle
# !kaggle datasets download -d andrewmvd/heart-failure-clinical-data
# !unzip heart-failure-clinical-data.zip

# 아래 코드로 데이터를 직접 확인하라.
# df.head()
# df.info()
# df.describe()
# df.shape

<details>
<summary>▶ 해답 확인</summary>

```python
df = pd.read_csv('heart_failure_clinical_records_dataset.csv')

print("=== 데이터 크기 ===")
print(f"  행: {df.shape[0]}개, 열: {df.shape[1]}개")
print()

print("=== 처음 5행 ===")
display(df.head())
print()

print("=== 데이터 타입 및 결측값 ===")
df.info()
print()

print("=== 기술 통계 ===")
display(df.describe())
```

**기대 결과:** 299행 × 13열, 결측값 없음(Non-Null Count = 299)

</details>

---

## Step 2. **탐색적 데이터 분석**(EDA)

### 문제 2. **타겟 변수**(Target Variable) 분포 확인

`DEATH_EVENT`(사망 여부)의 분포를 확인하라. 사망자와 생존자의 비율은 어떠한가?

In [ ]:
# DEATH_EVENT 분포를 확인하라.
# value_counts()와 시각화를 활용한다.


<details>
<summary>▶ 해답 확인</summary>

```python
print("=== 타겟 변수 분포 ===")
print(df['DEATH_EVENT'].value_counts())
print()
print(df['DEATH_EVENT'].value_counts(normalize=True).round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 막대 그래프
df['DEATH_EVENT'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'crimson'])
axes[0].set_title('DEATH_EVENT 분포')
axes[0].set_xlabel('0: 생존, 1: 사망')
axes[0].set_ylabel('환자 수')

# 파이 차트
df['DEATH_EVENT'].value_counts().plot(kind='pie', ax=axes[1],
    autopct='%1.1f%%', colors=['steelblue', 'crimson'],
    labels=['생존', '사망'])
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("→ 생존 203명(67.9%) vs 사망 96명(32.1%) — 불균형(Imbalanced) 데이터이다.")
```

</details>

### 문제 3. **수치형 변수**(Numerical Feature)의 분포 시각화

수치형 변수(`age`, `creatinine_phosphokinase`, `ejection_fraction`, `platelets`, `serum_creatinine`, `serum_sodium`, `time`)의 히스토그램을 그려라. 사망 여부(`DEATH_EVENT`)에 따라 색상을 구분하라.

In [ ]:
# 수치형 변수의 히스토그램을 그려라.
# Hint: seaborn의 histplot()에서 hue='DEATH_EVENT'를 사용한다.


<details>
<summary>▶ 해답 확인</summary>

```python
num_cols = ['age', 'creatinine_phosphokinase', 'ejection_fraction',
            'platelets', 'serum_creatinine', 'serum_sodium', 'time']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, hue='DEATH_EVENT', kde=True,
                 ax=axes[i], palette=['steelblue', 'crimson'])
    axes[i].set_title(f'{col} 분포')

# 빈 서브플롯 숨기기
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()
```

</details>

### 문제 4. **박스플롯**(Boxplot)으로 사망 여부별 수치 비교

`DEATH_EVENT`를 기준으로 주요 수치형 변수의 박스플롯을 그려라. 사망자와 생존자 간에 눈에 띄는 차이가 있는 변수는 무엇인가?

In [ ]:
# seaborn의 boxplot() 또는 violinplot()을 사용하라.
# Hint: hue='DEATH_EVENT'


<details>
<summary>▶ 해답 확인</summary>

```python
key_cols = ['age', 'ejection_fraction', 'serum_creatinine', 'serum_sodium', 'time']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for i, col in enumerate(key_cols):
    sns.boxplot(data=df, x='DEATH_EVENT', y=col, ax=axes[i],
                palette=['steelblue', 'crimson'])
    axes[i].set_title(col)
    axes[i].set_xlabel('0: 생존 / 1: 사망')

plt.tight_layout()
plt.show()

print("→ 관찰:")
print("  - time(관찰 기간): 사망자의 관찰 기간이 현저히 짧다.")
print("  - ejection_fraction(박출계수): 사망자의 박출계수가 낮은 경향이 있다.")
print("  - serum_creatinine(혈중 크레아틴): 사망자의 수치가 높은 경향이 있다.")
```

</details>

### 문제 5. **상관계수 히트맵**(Correlation Heatmap)

모든 변수 간의 **상관계수**(Correlation Coefficient)를 구하고 히트맵으로 시각화하라. `DEATH_EVENT`와 상관관계가 가장 높은 상위 3개 변수를 찾아라.

In [ ]:
# 상관계수 히트맵을 그려라.
# Hint: df.corr()과 sns.heatmap()


<details>
<summary>▶ 해답 확인</summary>

```python
corr = df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5)
plt.title('변수 간 상관계수 히트맵')
plt.tight_layout()
plt.show()

print("=== DEATH_EVENT와의 상관계수 (절대값 기준 상위 5개) ===")
death_corr = corr['DEATH_EVENT'].drop('DEATH_EVENT').abs().sort_values(ascending=False)
for col, val in death_corr.head(5).items():
    direction = "양(+)" if corr.loc[col, 'DEATH_EVENT'] > 0 else "음(-)"
    print(f"  {col:30s}: {corr.loc[col, 'DEATH_EVENT']:+.3f} ({direction} 상관)")
```

</details>

---

## Step 3. 모델 학습을 위한 **데이터 전처리**(Preprocessing)

### 문제 6. 수치형/범주형 변수 분리 및 **표준화**(StandardScaler)

수치형 입력 변수와 범주형 입력 변수를 분리하고, 수치형 변수에 `StandardScaler`를 적용하라. 출력 변수(타겟)는 `DEATH_EVENT`이다.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 수치형 변수, 범주형 변수, 타겟 변수를 분리하라.
# 수치형: age, creatinine_phosphokinase, ejection_fraction, platelets,
#         serum_creatinine, serum_sodium, time
# 범주형: anaemia, diabetes, high_blood_pressure, sex, smoking
# 타겟: DEATH_EVENT

# X_num = ___
# X_cat = ___
# y = ___

# StandardScaler로 수치형 변수를 표준화하라.
# scaler = ___
# X_num_scaled = ___

# 수치형(표준화 완료) + 범주형을 합쳐 최종 입력 데이터를 만들어라.
# X = ___


<details>
<summary>▶ 해답 확인</summary>

```python
from sklearn.preprocessing import StandardScaler

# 변수 분리
num_cols = ['age', 'creatinine_phosphokinase', 'ejection_fraction',
            'platelets', 'serum_creatinine', 'serum_sodium', 'time']
cat_cols = ['anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking']

X_num = df[num_cols]
X_cat = df[cat_cols]
y = df['DEATH_EVENT']

print(f"수치형 변수: {X_num.shape[1]}개 → {num_cols}")
print(f"범주형 변수: {X_cat.shape[1]}개 → {cat_cols}")
print(f"타겟 변수: {y.name}")
print()

# 표준화
scaler = StandardScaler()
X_num_scaled = pd.DataFrame(
    scaler.fit_transform(X_num),
    columns=num_cols,
    index=X_num.index
)

print("=== 표준화 전후 비교 (age) ===")
print(f"  전: 평균={X_num['age'].mean():.1f}, 표준편차={X_num['age'].std():.1f}")
print(f"  후: 평균={X_num_scaled['age'].mean():.4f}, 표준편차={X_num_scaled['age'].std():.4f}")
print()

# 합치기
X = pd.concat([X_num_scaled, X_cat], axis=1)
print(f"최종 입력 데이터: {X.shape}")
```

</details>

### 문제 7. **학습 데이터**(Train)와 **테스트 데이터**(Test) 분리

`train_test_split`을 사용하여 학습 데이터와 테스트 데이터를 8:2로 분리하라. 타겟의 비율을 유지하기 위해 `stratify` 매개변수를 사용하라.

In [ ]:
from sklearn.model_selection import train_test_split

# train_test_split으로 학습/테스트 데이터를 분리하라.
# X_train, X_test, y_train, y_test = ___

# 분리 결과를 확인하라.


<details>
<summary>▶ 해답 확인</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== 데이터 분리 결과 ===")
print(f"  학습 데이터: {X_train.shape[0]}개 ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"  테스트 데이터: {X_test.shape[0]}개 ({X_test.shape[0]/len(X)*100:.0f}%)")
print()
print(f"  학습 세트 사망 비율: {y_train.mean():.3f} ({y_train.sum()}/{len(y_train)})")
print(f"  테스트 세트 사망 비율: {y_test.mean():.3f} ({y_test.sum()}/{len(y_test)})")
print(f"  원본 사망 비율:      {y.mean():.3f}")
print()
print("→ stratify=y 덕분에 학습/테스트 모두 사망 비율이 약 32%로 유지된다.")
```

</details>

---

## Step 4. **분류 모델**(Classification Model) 학습

### 문제 8. **로지스틱 회귀**(Logistic Regression) 모델 학습 및 평가

`LogisticRegression`을 사용하여 모델을 학습하고, 테스트 데이터에 대한 예측 결과를 `classification_report`로 확인하라.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# LogisticRegression 모델을 생성하고 학습하라.
# model_lr = ___

# 테스트 데이터에 대해 예측하고 평가하라.
# pred_lr = ___
# print(classification_report(y_test, pred_lr))


<details>
<summary>▶ 해답 확인</summary>

```python
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train, y_train)

pred_lr = model_lr.predict(X_test)

print("=== Logistic Regression 평가 결과 ===")
print(classification_report(y_test, pred_lr, target_names=['생존(0)', '사망(1)']))

print("=== 오차 행렬(Confusion Matrix) ===")
cm = confusion_matrix(y_test, pred_lr)
print(f"  [[TN={cm[0,0]:2d}  FP={cm[0,1]:2d}]")
print(f"   [FN={cm[1,0]:2d}  TP={cm[1,1]:2d}]]")
print()
print(f"  정확도(Accuracy): {model_lr.score(X_test, y_test):.4f}")
```

</details>

### 문제 9. **교차 검증**(Cross-Validation)으로 안정적인 성능 평가

`cross_val_score`를 사용하여 **Stratified 5-Fold** 교차 검증을 수행하라. 단일 train/test split과 교차 검증 결과를 비교하라.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Stratified 5-Fold 교차 검증을 수행하라.
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# cv_scores = cross_val_score(___, ___, ___, cv=cv, scoring='accuracy')

# 결과를 출력하라.


<details>
<summary>▶ 해답 확인</summary>

```python
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression에 대해 교차 검증
model_cv = LogisticRegression(max_iter=1000, random_state=42)
cv_scores = cross_val_score(model_cv, X, y, cv=cv, scoring='accuracy')

print("=== Stratified 5-Fold 교차 검증 결과 ===")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: 정확도 = {score:.4f}")
print(f"\n  평균 정확도: {cv_scores.mean():.4f}")
print(f"  표준편차:    {cv_scores.std():.4f}")
print(f"  최종 보고:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print()

print("=== 단일 split vs 교차 검증 비교 ===")
print(f"  단일 split 정확도:       {model_lr.score(X_test, y_test):.4f}")
print(f"  5-Fold 평균 정확도:      {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print()
print("→ 교차 검증은 5번의 평가를 평균내므로 더 신뢰할 수 있는 성능 추정이다.")
```

</details>

---

## Step 5. **전처리 전후 비교**(Before/After Preprocessing)

### 문제 10. 전처리 없이 학습했을 때 vs 전처리 후 성능 비교

전처리(StandardScaler)를 적용하지 않은 원본 데이터와, 전처리를 적용한 데이터의 모델 성능을 비교하라. 전처리가 성능에 미치는 영향을 수치로 확인하라.

In [ ]:
# 전처리 없이 학습
# X_raw = pd.concat([X_num, X_cat], axis=1)  # 스케일링 안 한 원본
# X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
#     X_raw, y, test_size=0.2, random_state=42, stratify=y
# )

# 전처리 없이 Logistic Regression
# model_raw = ___
# 전처리 후 Logistic Regression (이미 학습한 model_lr)

# 두 모델의 정확도를 비교하라.


<details>
<summary>▶ 해답 확인</summary>

```python
# 전처리 없이 학습
X_raw = pd.concat([X_num, X_cat], axis=1)
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

model_raw = LogisticRegression(max_iter=1000, random_state=42)
model_raw.fit(X_train_raw, y_train_raw)
score_raw = model_raw.score(X_test_raw, y_test_raw)

# 전처리 후 결과 (이미 학습한 model_lr)
score_scaled = model_lr.score(X_test, y_test)

print("=== 전처리 전후 성능 비교 ===")
print(f"  전처리 없음: 정확도 = {score_raw:.4f}")
print(f"  전처리 적용: 정확도 = {score_scaled:.4f}")
print(f"  차이:        {score_scaled - score_raw:+.4f}")
print()

# 교차 검증으로도 비교
cv_raw = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_raw, y, cv=cv, scoring='accuracy'
)
cv_scaled = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X, y, cv=cv, scoring='accuracy'
)

print("=== 교차 검증 비교 ===")
print(f"  전처리 없음: {cv_raw.mean():.4f} ± {cv_raw.std():.4f}")
print(f"  전처리 적용: {cv_scaled.mean():.4f} ± {cv_scaled.std():.4f}")
```

</details>

---

# PART 4: **연습문제**(Exercises) — 12문항

---

### 연습문제 1. **머신러닝 유형 분류**

다음 각 문제를 **지도학습-분류**, **지도학습-회귀**, **비지도학습**, **강화학습** 중 하나로 분류하라.

(1) 이메일이 스팸인지 아닌지 분류  
(2) 내일의 주식 가격 예측  
(3) 고객을 구매 패턴에 따라 5개 그룹으로 나누기  
(4) 바둑 AI가 다음 최적의 수를 결정  
(5) 환자의 CT 사진에서 종양 유무 판별

<details>
<summary>▶ 해답 확인</summary>

```
(1) 지도학습-분류: 정답(스팸/아님)이 있고, 범주를 예측한다.
(2) 지도학습-회귀: 정답(가격)이 있고, 연속적인 수치를 예측한다.
(3) 비지도학습(군집화): 정답 없이 데이터의 구조(그룹)를 발견한다.
(4) 강화학습: 보상(승/패) 신호를 통해 최적의 행동을 학습한다.
(5) 지도학습-분류: 정답(종양 있음/없음)이 있고, 범주를 예측한다.
```

</details>

### 연습문제 2. **ML 워크플로우 순서 배열**

다음 6단계를 올바른 순서로 나열하라.

`모델 학습`, `데이터 전처리`, `모델 평가`, `문제 정의`, `배포`, `데이터 수집`

<details>
<summary>▶ 해답 확인</summary>

```
올바른 순서:
  1. 문제 정의 (Problem Definition)
  2. 데이터 수집 (Data Collection)
  3. 데이터 전처리 (Preprocessing)
  4. 모델 학습 (Model Training)
  5. 모델 평가 (Model Evaluation)
  6. 배포 (Deployment)
```

</details>

### 연습문제 3. **StandardScaler 수동 계산**

다음 데이터에 `StandardScaler`를 수동으로 적용하라.

```
데이터: [10, 20, 30, 40, 50]
```

(1) 평균과 표준편차를 구하라.  
(2) 각 값을 `z = (x - 평균) / 표준편차` 공식으로 변환하라.  
(3) 변환 후 평균과 표준편차를 확인하라.

In [ ]:
# 연습문제 3: 직접 코드를 작성하여 검증하라.
data = [10, 20, 30, 40, 50]

# (1) 평균과 표준편차
# mean = ___
# std = ___

# (2) 각 값을 변환
# scaled = ___

# (3) 변환 후 평균과 표준편차 확인


<details>
<summary>▶ 해답 확인</summary>

```python
import numpy as np

data = np.array([10, 20, 30, 40, 50])

# (1) 평균과 표준편차
mean = data.mean()    # 30.0
std = data.std()      # 14.142...

print(f"평균: {mean}")
print(f"표준편차: {std:.4f}")

# (2) 변환
scaled = (data - mean) / std
print(f"변환 결과: {scaled.round(4)}")
# [-1.4142, -0.7071, 0.0, 0.7071, 1.4142]

# (3) 검증
print(f"변환 후 평균: {scaled.mean():.6f}")  # ≈ 0
print(f"변환 후 표준편차: {scaled.std():.6f}")  # ≈ 1

# sklearn으로 검증
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
sklearn_result = scaler.fit_transform(data.reshape(-1, 1)).flatten()
print(f"sklearn 결과: {sklearn_result.round(4)}")
```

</details>

### 연습문제 4. **train_test_split 매개변수**

다음 코드의 각 매개변수가 하는 역할을 설명하라.

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

(1) `test_size=0.2`  
(2) `random_state=42`  
(3) `stratify=y`

<details>
<summary>▶ 해답 확인</summary>

```
(1) test_size=0.2
    → 전체 데이터의 20%를 테스트 세트로 사용한다.
    → 나머지 80%는 학습 세트가 된다.
    → 0.3이면 30%, 0.1이면 10%가 테스트 세트이다.

(2) random_state=42
    → 데이터를 무작위로 분할할 때 사용하는 시드(seed) 값이다.
    → 같은 숫자를 쓰면 매번 동일한 분할 결과를 얻는다.
    → 실험의 재현성(Reproducibility)을 보장한다.

(3) stratify=y
    → 분할 시 타겟 변수(y)의 비율을 유지한다.
    → 예: 원본에서 사망 32%이면, 학습/테스트 모두 약 32%가 된다.
    → 불균형 데이터에서 특히 중요하다.
```

</details>

### 연습문제 5. **K-Fold 교차 검증 이해**

5-Fold 교차 검증에서 전체 데이터가 100개일 때:

(1) 각 Fold의 테스트 세트 크기는 몇 개인가?  
(2) 각 Fold의 학습 세트 크기는 몇 개인가?  
(3) 모델은 총 몇 번 학습되는가?  
(4) 최종 성능은 어떻게 계산하는가?

<details>
<summary>▶ 해답 확인</summary>

```
(1) 각 Fold의 테스트 세트: 100 / 5 = 20개
(2) 각 Fold의 학습 세트: 100 - 20 = 80개
(3) 모델은 총 5번 학습된다. (각 Fold마다 1번)
(4) 최종 성능 = 5개 Fold의 성능 평균
    예: (0.82 + 0.87 + 0.84 + 0.79 + 0.86) / 5 = 0.836

    표준편차도 함께 보고한다.
    → 최종: 0.836 ± 0.028
    → 분산이 작을수록 모델이 안정적이라는 의미이다.
```

</details>

### 연습문제 6. **인코딩 방식 선택**

다음 각 변수에 적합한 인코딩 방식(**레이블 인코딩** 또는 **원-핫 인코딩**)을 선택하고 이유를 설명하라.

(1) 학점: A, B, C, D, F  
(2) 혈액형: A형, B형, O형, AB형  
(3) 티셔츠 사이즈: S, M, L, XL  
(4) 국적: 한국, 미국, 일본, 영국, 프랑스

<details>
<summary>▶ 해답 확인</summary>

```
(1) 학점 → 레이블 인코딩 (A=4, B=3, C=2, D=1, F=0)
    이유: 순서가 있다 (A > B > C > D > F). 숫자 크기에 의미가 있다.

(2) 혈액형 → 원-핫 인코딩
    이유: 순서가 없다. A형=0, B형=1이면 "B형이 A형보다 크다"는
    잘못된 의미가 부여될 수 있다.

(3) 티셔츠 사이즈 → 레이블 인코딩 (S=0, M=1, L=2, XL=3)
    이유: 순서가 있다 (S < M < L < XL).

(4) 국적 → 원-핫 인코딩
    이유: 순서가 없다. 한국=0, 미국=1이면 "미국이 한국보다 크다"는
    잘못된 의미가 부여된다.
```

</details>

### 연습문제 7. **Scikit-learn API 패턴 코딩**

다음 코드의 빈칸을 채워 `MinMaxScaler`를 사용하라.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

data = np.array([[100], [200], [300], [400], [500]])

# 1) MinMaxScaler 객체를 생성하라.
# scaler = ___

# 2) fit_transform()으로 변환하라.
# data_scaled = ___

# 3) 변환 결과를 출력하라. (0~1 범위여야 한다)
# print(data_scaled)

# 4) 새로운 데이터 [[250]]을 transform()으로 변환하라.
# new_data = ___
# print(f"250 → {new_data}")


<details>
<summary>▶ 해답 확인</summary>

```python
from sklearn.preprocessing import MinMaxScaler
import numpy as np

data = np.array([[100], [200], [300], [400], [500]])

# 1) 객체 생성
scaler = MinMaxScaler()

# 2) fit_transform
data_scaled = scaler.fit_transform(data)

# 3) 결과 출력
print("변환 결과:")
print(data_scaled)
# [[0.  ], [0.25], [0.5 ], [0.75], [1.  ]]

# 4) 새로운 데이터 변환
new_data = scaler.transform([[250]])
print(f"250 → {new_data[0][0]:.4f}")  # 0.375
```

</details>

### 연습문제 8. **Stratified K-Fold 구현**

심부전증 데이터(또는 임의 데이터)에 대해 `StratifiedKFold`를 직접 사용하여 3-Fold 교차 검증을 수행하라. `cross_val_score`를 쓰지 말고, `for` 루프를 사용하여 직접 구현하라.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

# 예제 데이터 (또는 심부전증 데이터의 X, y 사용)
from sklearn.datasets import make_classification
X_ex, y_ex = make_classification(n_samples=150, n_features=5, random_state=42)

# StratifiedKFold를 사용하여 직접 교차 검증을 구현하라.
# skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
# scores = []
# for fold, (train_idx, test_idx) in enumerate(skf.split(X_ex, y_ex), 1):
#     ___


<details>
<summary>▶ 해답 확인</summary>

```python
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

from sklearn.datasets import make_classification
X_ex, y_ex = make_classification(n_samples=150, n_features=5, random_state=42)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_ex, y_ex), 1):
    X_tr, X_te = X_ex[train_idx], X_ex[test_idx]
    y_tr, y_te = y_ex[train_idx], y_ex[test_idx]
    
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_tr, y_tr)
    
    pred = model.predict(X_te)
    acc = accuracy_score(y_te, pred)
    scores.append(acc)
    
    print(f"Fold {fold}: 학습={len(train_idx)}, 테스트={len(test_idx)}, 정확도={acc:.4f}")

print(f"\n평균 정확도: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
```

</details>

### 연습문제 9. **전처리 파이프라인 완성**

다음 코드는 심부전증 데이터의 전체 전처리 파이프라인이다. 빈칸을 채워 완성하라.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression

# 1단계: 데이터 불러오기
# df = pd.read_csv('heart_failure_clinical_records_dataset.csv')

# 2단계: 특성과 타겟 분리
# num_cols = ___
# cat_cols = ___
# X_num = df[num_cols]
# X_cat = df[cat_cols]
# y = df['DEATH_EVENT']

# 3단계: 스케일링
# scaler = ___
# X_num_scaled = pd.DataFrame(scaler.fit_transform(X_num), columns=num_cols)

# 4단계: 합치기
# X = pd.concat([X_num_scaled, X_cat.reset_index(drop=True)], axis=1)

# 5단계: 학습/테스트 분리
# X_train, X_test, y_train, y_test = ___

# 6단계: 모델 학습 및 평가
# model = ___
# model.fit(___, ___)
# print(f"정확도: {model.score(X_test, y_test):.4f}")

# 7단계: 교차 검증
# cv_scores = cross_val_score(___, ___, ___, cv=5, scoring='accuracy')
# print(f"CV 정확도: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


<details>
<summary>▶ 해답 확인</summary>

```python
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression

# 1단계
df = pd.read_csv('heart_failure_clinical_records_dataset.csv')

# 2단계
num_cols = ['age', 'creatinine_phosphokinase', 'ejection_fraction',
            'platelets', 'serum_creatinine', 'serum_sodium', 'time']
cat_cols = ['anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking']
X_num = df[num_cols]
X_cat = df[cat_cols]
y = df['DEATH_EVENT']

# 3단계
scaler = StandardScaler()
X_num_scaled = pd.DataFrame(scaler.fit_transform(X_num), columns=num_cols)

# 4단계
X = pd.concat([X_num_scaled, X_cat.reset_index(drop=True)], axis=1)

# 5단계
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6단계
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
print(f"정확도: {model.score(X_test, y_test):.4f}")

# 7단계
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"CV 정확도: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
```

</details>

### 연습문제 10. **scoring 매개변수 변경**

`cross_val_score`의 `scoring` 매개변수를 `'accuracy'` 대신 다음 지표로 바꿔서 실행하고 결과를 비교하라.

(1) `'precision'`  
(2) `'recall'`  
(3) `'f1'`  
(4) `'roc_auc'`

각 지표의 의미와, 심부전증 예측에서 어떤 지표가 가장 중요한지 자신의 의견을 작성하라.

In [ ]:
# 다양한 scoring으로 교차 검증을 수행하라.
# scoring_list = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
# for scoring in scoring_list:
#     scores = cross_val_score(___, ___, ___, cv=5, scoring=scoring)
#     print(f"  {scoring:12s}: {scores.mean():.4f} ± {scores.std():.4f}")


<details>
<summary>▶ 해답 확인</summary>

```python
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression(max_iter=1000, random_state=42)

scoring_list = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

print("=== 다양한 평가 지표 비교 ===")
for scoring in scoring_list:
    scores = cross_val_score(model, X, y, cv=cv, scoring=scoring)
    print(f"  {scoring:12s}: {scores.mean():.4f} ± {scores.std():.4f}")

print()
print("=== 각 지표의 의미 ===")
print("  accuracy:  전체 중 맞힌 비율")
print("  precision: '사망'이라고 예측한 것 중 실제 사망 비율 (오경보 줄이기)")
print("  recall:    실제 사망자 중 모델이 잡아낸 비율 (놓침 줄이기)")
print("  f1:        precision과 recall의 조화 평균")
print("  roc_auc:   모델의 전반적 분류 능력")
print()
print("→ 심부전증 예측에서는 recall이 가장 중요하다.")
print("   이유: 실제 사망 위험 환자를 놓치면(FN) 생명이 위험하다.")
print("   오경보(FP)는 추가 검사로 해결 가능하지만, 놓침(FN)은 치명적이다.")
```

</details>

### 연습문제 11. 스케일링 방법에 따른 성능 비교

`StandardScaler`와 `MinMaxScaler`를 각각 적용한 후 교차 검증 성능을 비교하라. 어떤 스케일러가 더 좋은 성능을 보이는가?

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# StandardScaler와 MinMaxScaler를 각각 적용하여 비교하라.


<details>
<summary>▶ 해답 확인</summary>

```python
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    '스케일링 없음': None
}

for name, scl in scalers.items():
    if scl is not None:
        X_num_s = pd.DataFrame(scl.fit_transform(X_num), columns=num_cols)
    else:
        X_num_s = X_num.reset_index(drop=True)
    
    X_combined = pd.concat([X_num_s, X_cat.reset_index(drop=True)], axis=1)
    
    model = LogisticRegression(max_iter=1000, random_state=42)
    scores = cross_val_score(model, X_combined, y, cv=cv, scoring='accuracy')
    print(f"  {name:18s}: {scores.mean():.4f} ± {scores.std():.4f}")
```

</details>

### 연습문제 12 (도전). **Pipeline을 사용한 전처리 자동화**

Scikit-learn의 `Pipeline`을 사용하면 전처리와 모델 학습을 하나의 객체로 묶을 수 있다. `Pipeline`을 사용하여 `StandardScaler → LogisticRegression` 파이프라인을 만들고 교차 검증하라.

In [ ]:
from sklearn.pipeline import Pipeline

# Pipeline을 구성하라.
# pipe = Pipeline([
#     ('scaler', ___),
#     ('model', ___)
# ])

# 교차 검증을 수행하라.
# scores = cross_val_score(pipe, ___, ___, cv=5, scoring='accuracy')


<details>
<summary>▶ 해답 확인</summary>

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Pipeline 구성
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

# 수치형 + 범주형 합친 원본 데이터 사용
X_all = pd.concat([X_num, X_cat], axis=1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X_all, y, cv=cv, scoring='accuracy')

print("=== Pipeline + 교차 검증 결과 ===")
for i, s in enumerate(scores, 1):
    print(f"  Fold {i}: {s:.4f}")
print(f"\n  평균: {scores.mean():.4f} ± {scores.std():.4f}")
print()
print("→ Pipeline의 장점:")
print("  1. 전처리와 모델을 하나로 묶어 코드가 간결하다.")
print("  2. 교차 검증 시 각 Fold에서 전처리가 독립적으로 수행된다.")
print("     (데이터 누수(Data Leakage) 방지)")
```

</details>

---

## 핵심 요약

1. **머신러닝**(Machine Learning)은 데이터로부터 패턴을 학습하여 예측하는 알고리즘이다.
2. ML **워크플로우**는 문제 정의 → 데이터 수집 → **전처리** → 모델 학습 → **모델 평가** → 배포 순서로 진행된다.
3. **전처리**의 3대 핵심: 결측값 처리, 스케일링(StandardScaler, MinMaxScaler), 인코딩(Label, OneHot)이다.
4. **교차 검증**(Cross-Validation)은 모델 성능을 안정적으로 평가하는 방법이며, 불균형 데이터에서는 **Stratified K-Fold**를 사용해야 한다.
5. Scikit-learn의 공통 패턴: `fit()` → `transform()` / `predict()`를 따른다.
6. **Pipeline**을 사용하면 전처리와 모델 학습을 하나로 묶어 데이터 누수를 방지할 수 있다.

---

*다음 주차(7주차)에서는 **결정 트리**(Decision Tree), **평가 지표**(Metrics), **랜덤 포레스트**(Random Forest)를 다룬다.*